# Fase 1 — Baseline Tabular: MC Control vs Q-Learning

**Curso:** DS5345 · Aprendizaje por Refuerzo (UTEC 2026-II)  
**Tarea restringida:** Persecución / intercepción de balón (RoboCup 2D)  

### Objetivos P1 cubiertos en este cuaderno
1. Formular el MDP $\langle \mathcal{S},\mathcal{A},\mathcal{P},\mathcal{R},\gamma\rangle$ con discretización justificada.
2. Entrenar **Monte Carlo Control On-Policy (First-Visit)** y **Q-Learning** tabular.
3. Comparar exploración **$\varepsilon$ fijo** vs **$\varepsilon$ decreciente**.
4. Reportar retorno $G_0$, **tasa de éxito temporal** y pasos por episodio; visualizar $V^*$ y $\pi^*$.
5. (Opcional) Verificar conexión Docker → `rcssserver` con el cliente UDP.


In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# Resolver src/ tanto en host como dentro del contenedor
CANDIDATES = [
    Path.cwd() / "src",
    Path.cwd().parent / "src",
    Path("/workspace/src"),
]
for p in CANDIDATES:
    if p.is_dir() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

from agents.mc_control import train_mc_control
from agents.q_learning import train_q_learning
from env import BallPursuitSimEnv
from metrics import summarize_run
from plotting import (
    plot_learning_curves,
    plot_trajectory,
    plot_value_and_policy,
    rollout_greedy,
)

np.random.seed(42)
print("Módulos cargados.")


## 1. Modelado MDP y discretización

| Componente | Definición |
|---|---|
| $\mathcal{S}$ | $(d_b, \theta_b)$ discretizados: 4 bins de distancia × 5 de ángulo → $|\mathcal{S}|=20$ |
| $\mathcal{A}$ | `{DASH 100, DASH 50, TURN +35°, TURN −35°}` → $|\mathcal{A}|=4$ |
| $\mathcal{R}$ | $+100$ si $d_b<0.8$; $-20$ timeout; $-1 + 5\Delta d$ en pasos intermedios |
| $\gamma$ | $0.99$; horizonte máx. $80$ pasos |
| $\mathcal{P}$ | Dinámica cinemática determinista del proxy (despliegue real vía `rcssserver`) |

**Justificación analítica de bins:** el umbral $0.8\,\mathrm{m}$ coincide con el *kickable area* de rcssserver; $3\,\mathrm{m}$ y $8\,\mathrm{m}$ separan reacción próxima/media/lejana; $\pm 15^\circ$ es el cono frontal para dash efectivo y $\pm 60^\circ$ diferencia giros parciales de giros completos. Cardinalidad de $Q$: $|\mathcal{S}|\cdot|\mathcal{A}|=80$ pares, compatible con exploración tabular en pocos miles de episodios.


## 2. Entrenamiento comparativo (4 configuraciones)

- MC Control + $\varepsilon$ decay  
- MC Control + $\varepsilon$ fijo $=0.1$  
- Q-Learning ($\alpha=0.1$) + $\varepsilon$ decay  
- Q-Learning + $\varepsilon$ fijo $=0.1$


In [ ]:
N_EPISODES = 3500
SEED = 42

configs = [
    ("MC | ε decay", train_mc_control, {"exploration": "decay"}),
    ("MC | ε fijo=0.1", train_mc_control, {"exploration": "fixed", "eps_fixed": 0.1}),
    ("QL | ε decay", train_q_learning, {"exploration": "decay", "alpha": 0.1}),
    ("QL | ε fijo=0.1", train_q_learning, {"exploration": "fixed", "eps_fixed": 0.1, "alpha": 0.1}),
]

results = {}
summaries = {}
for name, trainer, kwargs in configs:
    print(f"Entrenando: {name} ...")
    env = BallPursuitSimEnv(seed=SEED)
    res = trainer(env=env, n_episodes=N_EPISODES, seed=SEED, **kwargs)
    results[name] = res
    summaries[name] = summarize_run(res, last_n=100)
    s = summaries[name]
    print(
        f"  G0={s['mean_return_last']:.1f} | pasos={s['mean_steps_last']:.1f} | "
        f"éxito(últ.100)={100*s['success_rate_last']:.1f}%"
    )


## 3. Curvas de aprendizaje, tasa de éxito y eficiencia


In [ ]:
fig = plot_learning_curves(results, window=50)
plt.show()

print("\nResumen (últimos 100 episodios):")
for name, s in summaries.items():
    print(f"  {name:18s}  G0={s['mean_return_last']:7.1f}  "
          f"pasos={s['mean_steps_last']:5.1f}  éxito={100*s['success_rate_last]:5.1f}%")


## 4. Política y valor de la mejor configuración


In [ ]:
best_name = max(summaries, key=lambda k: summaries[k]["success_rate_last"])
best = results[best_name]
print("Mejor configuración:", best_name)

plot_value_and_policy(best["Q"], title=f"Mejor política: {best_name}")
plt.show()

env_vis = BallPursuitSimEnv(seed=7)
rollout_greedy(env_vis, best["Q"])
plot_trajectory(env_vis, title=f"Trayectoria greedy — {best_name}")
plt.show()
print("Éxito en rollout:", env_vis.success, "| pasos:", env_vis.steps)


## 5. Verificación Docker / rcssserver (smoke test)

Ejecutar solo si `docker compose up -d` está activo. Si no hay servidor, la celda falla de forma controlada.


In [ ]:
import os

try:
    from robocup_client import RoboCup2DClient
    host = os.getenv("SERVER_HOST", "127.0.0.1")
    port = int(os.getenv("SERVER_PORT", "6000"))
    client = RoboCup2DClient(host=host, port=port, team_name="UTEC_P1")
    ok = client.connect(init_pos=(-10.0, 0.0))
    if ok:
        obs = client.get_latest_observation()
        print("Conectado a rcssserver. Observación:", obs)
        client.render_field(title="P1 — Verificación de conexión")
        client.close()
    else:
        print("No se pudo conectar. ¿Está corriendo `docker compose up -d`?")
except Exception as e:
    print(f"Smoke test omitido/falló: {e}")
